# Module 10 - 실습 3 솔루션: AST 기반 코드 압축

In [ ]:
import sys
sys.path.insert(0, '../..')

import ast
from dataclasses import dataclass

print("AST 코드 압축 솔루션 준비 완료")

## 실습 1 솔루션: AST 기초

In [ ]:
sample_code = '''
class Calculator:
    """계산기 클래스."""
    
    def add(self, a: int, b: int) -> int:
        """두 수를 더합니다."""
        result = a + b
        return result
    
    def multiply(self, a: int, b: int) -> int:
        return a * b
'''

tree = ast.parse(sample_code)
classes = []
functions = []

for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef):
        classes.append(node.name)
    elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        functions.append(node.name)

print(f"발견된 클래스: {classes}")
print(f"발견된 함수: {functions}")

In [ ]:
assert "Calculator" in classes
assert "add" in functions
assert "multiply" in functions
print("✅ 실습 1 완료! AST 파싱이 올바르게 작동합니다.")

## 실습 2 솔루션: PythonCodeCompressor

In [ ]:
class PythonCodeCompressor:
    """Python 코드의 구조적 압축."""
    
    def extract_signatures(self, source_code: str) -> str:
        """Python 코드에서 클래스와 함수 시그니처를 추출합니다."""
        try:
            tree = ast.parse(source_code)
        except SyntaxError:
            return source_code
        
        lines = []
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef):
                # 부모 클래스 목록
                bases = ", ".join(
                    ast.unparse(b) for b in node.bases
                ) if node.bases else ""
                lines.append(f"class {node.name}({bases}):")
                # 클래스 docstring
                docstring = ast.get_docstring(node)
                if docstring:
                    lines.append(f'    """{docstring}"""')
            
            elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                prefix = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
                args = self._get_args(node)
                returns = self._get_return_annotation(node)
                sig = f"    {prefix} {node.name}({args}){returns}:"
                
                docstring = ast.get_docstring(node)
                if docstring:
                    lines.append(sig)
                    lines.append(f'        """{docstring}"""')
                    lines.append("        ...")
                else:
                    lines.append(sig)
                    lines.append("        ...")
        
        return "\n".join(lines)
    
    def _get_args(self, node) -> str:
        """함수 인자를 문자열로 반환합니다."""
        args = []
        for arg in node.args.args:
            annotation = ""
            if arg.annotation:
                annotation = f": {ast.unparse(arg.annotation)}"
            args.append(f"{arg.arg}{annotation}")
        return ", ".join(args)
    
    def _get_return_annotation(self, node) -> str:
        """반환 타입 어노테이션을 반환합니다."""
        if node.returns:
            return f" -> {ast.unparse(node.returns)}"
        return ""

In [ ]:
compressor = PythonCodeCompressor()
summary = compressor.extract_signatures(sample_code)
print("추출된 시그니처:")
print(summary)

In [ ]:
assert "Calculator" in summary
assert "add" in summary
assert "multiply" in summary
assert "result = a + b" not in summary
assert len(summary) < len(sample_code)

print(f"원본 크기: {len(sample_code)}자")
print(f"요약 크기: {len(summary)}자")
print(f"압축률: {(1 - len(summary)/len(sample_code))*100:.0f}%")
print("✅ 실습 2 완료! PythonCodeCompressor가 올바르게 작동합니다.")

## 실습 3 솔루션: 압축 효과 측정

In [ ]:
large_code = '''
import logging
import time
from pathlib import Path

logger = logging.getLogger(__name__)


class DataProcessor:
    """데이터를 처리하는 클래스."""

    def __init__(self, config: dict):
        self.config = config
        self.logger = logging.getLogger(__name__)
        self.cache = {}
        self._initialized = False

    def process(self, data: list[dict]) -> list[dict]:
        """데이터를 처리합니다."""
        results = []
        for item in data:
            validated = self._validate(item)
            if validated:
                transformed = self._transform(validated)
                results.append(transformed)
        self.logger.info(f"Processed {len(results)} items")
        return results

    def _validate(self, item: dict) -> dict | None:
        """항목을 검증합니다."""
        if "id" not in item:
            self.logger.warning("Missing id field")
            return None
        if "value" not in item:
            self.logger.warning("Missing value field")
            return None
        return item

    def _transform(self, item: dict) -> dict:
        """항목을 변환합니다."""
        return {
            "id": item["id"],
            "processed_value": item["value"] * 2,
            "timestamp": time.time(),
        }
'''

summary = compressor.extract_signatures(large_code)

print(f"원본 코드 크기: {len(large_code)}자")
print(f"압축 후 크기: {len(summary)}자")
print(f"\n압축된 코드:")
print(summary)

In [ ]:
assert summary is not None
assert len(summary) < len(large_code) * 0.7
assert "DataProcessor" in summary
assert "self.cache = {}" not in summary

compression_ratio = (1 - len(summary)/len(large_code)) * 100
print(f"압축률: {compression_ratio:.0f}%")
print("✅ 실습 3 완료! AST 기반 코드 압축이 올바르게 작동합니다.")